# 1-2. 산포 통계량

- 분산, 표준편차, 변동계수, 스케일링, 범위와 IQR을 계산해본다.

In [1]:
# 셀 목적: 산포통계량 계산에 필요한 NumPy·SciPy·Pandas를 불러온다.
# 해석 포인트: 같은 분산과 표준편차도 라이브러리마다 기본 ddof가 다를 수 있으므로 인자를 확인한다.
import numpy as np  # 배열과 평균·분산 등 수치 계산을 위해 NumPy를 np로 불러옴
from scipy import stats  # 정규성·t·ANOVA·상관·카이제곱 검정을 위해 scipy.stats를 불러옴
import pandas as pd  # 표 형태 데이터 처리를 위해 Pandas를 pd로 불러옴

## 1. 분산 계산

In [2]:
# 셀 목적: 동일한 데이터로 표본분산과 모분산을 계산해 분모의 차이를 비교한다.
# 해석 포인트: ddof=1은 n-1로 나누는 표본분산, ddof=0은 n으로 나누는 모분산이다.
# 분산을 계산할 데이터
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# ddof=1: 표본분산, 분모가 n-1
print(np.var(x, ddof=1))  # x의 표본분산을 출력하며, ddof=1이므로 분모는 n-1을 사용

# ddof를 생략하면 0: 모분산, 분모가 n
print(np.array(x).var())  # x를 NumPy 배열로 바꾼 뒤 기본 설정(ddof=0)으로 계산한 모분산을 출력

# Pandas Series로 모분산 계산
print(pd.Series(x).var(ddof=0))  # x를 Pandas Series로 바꾼 뒤 ddof=0으로 계산한 모분산을 출력

2.5
2.0
2.0


In [3]:
# 셀 목적: 평균에서 각 값까지의 거리를 제곱해 분산을 직접 계산한다.
# 해석 포인트: 편차를 그대로 합하면 0이 되므로 제곱한 편차의 평균으로 퍼짐을 측정한다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

mean = np.mean(x)              # 평균: 3
squared_deviations = (np.array(x) - mean) ** 2  # 각 관측값의 평균 편차를 제곱
variance = np.mean(squared_deviations)  # 편차 제곱의 평균으로 모분산을 계산

print(squared_deviations)  # [4. 1. 0. 1. 4.]
print(variance)           # 2.0

[4. 1. 0. 1. 4.]
2.0


### np.var() 계산 과정 확인

`np.var(x)`가 분산을 계산하는 과정을 직접 풀어서 확인한다.

In [4]:
# 셀 목적: 평균 → 편차 제곱 → 편차 제곱의 평균 순서로 모분산 공식을 재현한다.
# 해석 포인트: 직접 계산한 값과 np.var() 결과가 같은지 확인해 공식과 함수의 연결을 검증한다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 1. 데이터의 평균 계산
mean = np.mean(x)  # 데이터의 산술평균을 계산

# 2. 각 값에서 평균을 뺀 뒤 제곱
squared_deviations = (np.array(x) - mean) ** 2  # 각 관측값의 평균 편차를 제곱

# 3. 제곱한 편차들의 평균 = 모분산
variance = np.mean(squared_deviations)  # 편차 제곱의 평균으로 모분산을 계산

print(f"평균: {mean}")  # 분산 계산의 기준점이 되는 x의 산술평균을 출력
print(f"편차 제곱: {squared_deviations}")  # 각 관측값에서 평균을 뺀 편차를 제곱한 값들을 출력
print(f"직접 계산한 분산: {variance}")  # 편차 제곱의 평균으로 직접 계산한 x의 모분산을 출력
print(f"np.var()로 계산한 분산: {np.var(x)}")  # np.var()로 계산한 모분산을 출력해 직접 계산한 결과와 비교

평균: 3.0
편차 제곱: [4. 1. 0. 1. 4.]
직접 계산한 분산: 2.0
np.var()로 계산한 분산: 2.0


### ddof의 의미

분산은 편차 제곱의 합을 `(데이터 개수 - ddof)`로 나눈 값이다.

- `ddof=0`: 데이터 개수 `n`으로 나눔 → 모분산
- `ddof=1`: 데이터 개수 `n-1`로 나눔 → 표본분산

표본으로 모집단의 분산을 추정할 때는 분산이 작게 계산되는 경향을 보정하기 위해 보통 `ddof=1`을 사용한다.

In [5]:
# 셀 목적: 같은 편차제곱합을 n과 n-1로 각각 나누어 모분산과 표본분산을 비교한다.
# 해석 포인트: 표본으로 모집단 분산을 추정할 때는 자유도 보정을 위해 n-1을 사용한다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 평균에서 떨어진 정도를 제곱한 값들의 합
squared_deviation_sum = np.sum((np.array(x) - np.mean(x)) ** 2)  # 평균 편차를 제곱한 값들의 합을 계산

# 모분산: 데이터 개수 n으로 나눔
population_variance = squared_deviation_sum / len(x)  # 편차제곱합을 n으로 나누어 모분산을 계산

# 표본분산: 데이터 개수 n - 1로 나눔
sample_variance = squared_deviation_sum / (len(x) - 1)  # 편차제곱합을 n-1로 나누어 표본분산을 계산

print(f"편차 제곱의 합: {squared_deviation_sum}")  # 모분산과 표본분산의 공통 분자인 편차 제곱합을 출력
print(f"모분산 (ddof=0): {population_variance}")  # 편차 제곱합을 자료 수 n으로 나눈 모분산을 출력
print(f"표본분산 (ddof=1): {sample_variance}")  # 편차 제곱합을 n-1로 나눈 불편추정량인 표본분산을 출력

print(f"np.var(x, ddof=0): {np.var(x, ddof=0)}")  # NumPy에서 ddof=0으로 계산한 모분산을 출력
print(f"np.var(x, ddof=1): {np.var(x, ddof=1)}")  # NumPy에서 ddof=1로 계산한 표본분산을 출력

편차 제곱의 합: 10.0
모분산 (ddof=0): 2.0
표본분산 (ddof=1): 2.5
np.var(x, ddof=0): 2.0
np.var(x, ddof=1): 2.5


### Pandas Series로 분산 계산

`Series`는 Pandas의 1차원 데이터 구조이다.
리스트를 `Series`로 바꾸면 Pandas 방식의 통계 함수를 사용할 수 있다.

In [6]:
# 셀 목적: Python 리스트를 Pandas Series로 변환한 뒤 분산 계산 방식을 확인한다.
# 해석 포인트: Series.var()의 기본값은 ddof=1이므로 모분산이 필요하면 ddof=0을 명시한다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 리스트를 Pandas Series 형태로 변환
series = pd.Series(x)  # 분산·표준편차 메서드를 사용할 Pandas Series 생성

print(series)  # 분산 계산에 사용할 Pandas Series의 값과 인덱스를 출력

# ddof=0: 모분산, n으로 나눔
print(f"모분산: {series.var(ddof=0)}")  # Pandas Series에서 ddof=0으로 계산한 모분산을 출력

# ddof=1: 표본분산, n-1로 나눔
print(f"표본분산: {series.var(ddof=1)}")  # Pandas Series에서 ddof=1로 계산한 표본분산을 출력

0    1
1    2
2    3
3    4
4    5
dtype: int64
모분산: 2.0
표본분산: 2.5


## 2. 표준편차 계산

In [ ]:
# 셀 목적: NumPy 배열과 Pandas Series에서 표준편차를 계산한다.
# 해석 포인트: 표준편차는 분산의 제곱근이어서 원자료와 같은 단위로 퍼짐을 해석할 수 있다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 표본표준편차: 분모가 n - 1
print(np.std(x, ddof=1))  # NumPy로 계산한 x의 표본표준편차(ddof=1)를 출력

# 모표준편차: 분모가 n
print(np.array(x).std(ddof=0))  # NumPy 배열 메서드로 계산한 x의 모표준편차(ddof=0)를 출력

# Pandas Series로 표본표준편차 계산
print(pd.Series(x).std(ddof=1))  # Pandas Series로 계산한 x의 표본표준편차(ddof=1)를 출력

### 표준편차의 계산 과정

표준편차는 분산에 제곱근을 씌운 값이다.
분산은 값의 단위가 제곱되므로, 실제 데이터 단위로 다시 보기 위해 제곱근을 사용한다.

표준편차 = √분산
### 표준편차란?

분산은 데이터가 평균에서 얼마나 퍼져 있는지 나타내지만,
값이 제곱되어 있어서 바로 이해하기 어렵다.

그래서 분산에 제곱근을 씌워 원래 데이터와 같은 기준으로 만든 값이 표준편차다.

예를 들어 점수의 표준편차가 1.41이라면,
점수들이 평균에서 대략 1.41점 정도 떨어져 있다고 이해할 수 있다.

In [ ]:
# 셀 목적: 모분산의 제곱근을 직접 구해 np.std()의 모표준편차와 비교한다.
# 해석 포인트: 두 결과가 같으면 표준편차=√분산 관계가 코드에서도 확인된 것이다.
x = [1, 2, 3, 4, 5]  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 모분산: ddof=0이므로 n으로 나눔
population_variance = np.var(x, ddof=0)  # 편차제곱합을 n으로 나누어 모분산을 계산

# 모표준편차: 모분산의 제곱근
population_std = np.sqrt(population_variance)  # 모분산의 제곱근으로 모표준편차를 계산

print(f"모분산: {population_variance}")  # 표준편차를 직접 계산하기 전에 구한 x의 모분산을 출력
print(f"직접 계산한 모표준편차: {population_std}")  # 모분산의 양의 제곱근으로 직접 계산한 모표준편차를 출력
print(f"np.std()로 계산한 모표준편차: {np.std(x, ddof=0)}")  # np.std()로 계산한 모표준편차를 출력해 직접 계산값과 비교

## 3. 변동계수의 필요성

표준편차는 데이터의 크기에 영향을 받는다.
변동계수는 평균에 비해 데이터가 얼마나 퍼져 있는지 나타낸다.

변동계수 = 표준편차 / 평균

In [7]:
# 셀 목적: 모든 값을 10배 했을 때 표준편차가 어떻게 변하는지 확인한다.
# 해석 포인트: 원자료의 단위가 10배 커지면 표준편차도 10배 커지므로 단위가 다른 변수끼리 직접 비교하기 어렵다.
# x2는 x1의 모든 값을 10배로 만든 데이터
x1 = np.array([1, 2, 3, 4, 5])  # 첫 번째 비교용 분포 또는 기준 배열을 x1에 저장
x2 = x1 * 10  # 두 번째 비교용 분포 또는 배율 변경 배열을 x2에 저장

# 표본표준편차를 비교
print(np.std(x1, ddof=1))  # 원자료 x1의 표본표준편차를 출력
print(np.std(x2, ddof=1))  # x1을 10배 한 x2의 표본표준편차를 출력해 단위 변화의 영향을 확인

1.5811388300841898
15.811388300841896


## 4. 변동계수 계산

변동계수는 표준편차를 평균으로 나눈 값이다.
값이 작을수록 평균에 비해 데이터가 덜 퍼져 있다.

In [8]:
# 셀 목적: SciPy의 variation()으로 두 데이터의 변동계수를 계산한다.
# 해석 포인트: 변동계수는 표준편차/평균이므로 값의 단위나 전체 배율이 달라도 상대적 산포를 비교할 수 있다.
# scipy의 변동계수 함수
print(stats.variation(x1))  # SciPy로 계산한 x1의 변동계수(표준편차/평균)를 출력
print(stats.variation(x2))  # SciPy로 계산한 x2의 변동계수를 출력해 척도 변화에도 값이 유지되는지 확인

0.47140452079103173
0.4714045207910317


In [9]:
# 셀 목적: 변동계수 공식을 직접 계산해 stats.variation()의 결과와 비교한다.
# 해석 포인트: 함수 기본 설정과 맞추기 위해 모표준편차인 ddof=0을 사용한다.
# stats.variation()은 기본적으로 ddof=0인 표준편차를 사용한다.
x1_coefficient = np.std(x1, ddof=0) / np.mean(x1)  # x1의 모표준편차를 평균으로 나누어 변동계수 계산
x2_coefficient = np.std(x2, ddof=0) / np.mean(x2)  # x2의 모표준편차를 평균으로 나누어 변동계수 계산

print(x1_coefficient)  # 표준편차를 평균으로 나누어 직접 계산한 x1의 변동계수를 출력
print(x2_coefficient)  # 표준편차를 평균으로 나누어 직접 계산한 x2의 변동계수를 출력

0.47140452079103173
0.4714045207910317


## 5. 스케일링

스케일링은 크기 범위가 다른 데이터를 비교하거나
머신러닝 모델에 넣기 쉽게 값을 변환하는 작업이다.

In [10]:
# 셀 목적: 스케일링 실습에 사용할 원자료 x1과 10배 확대 자료 x2를 준비한다.
# 해석 포인트: 두 배열은 절대 크기는 다르지만 상대적인 위치 구조는 동일하다.
x1 = np.array([1, 2, 3, 4, 5])  # 첫 번째 비교용 분포 또는 기준 배열을 x1에 저장
x2 = x1 * 10  # 두 번째 비교용 분포 또는 배율 변경 배열을 x2에 저장

### 5-1. Standard Scaling

각 값이 평균에서 표준편차 기준으로 얼마나 떨어져 있는지 나타낸다.

In [11]:
# 셀 목적: 각 값을 평균 0·표준편차 1 기준의 z-score로 표준화한다.
# 해석 포인트: 배율이 다른 x1과 x2가 표준화 후 같은 값이 되는지 확인한다.
# (값 - 평균) / 표준편차
z1 = (x1 - x1.mean()) / x1.std()  # x1의 각 값을 평균 0·표준편차 1의 z-score로 변환
z2 = (x2 - x2.mean()) / x2.std()  # x2의 각 값을 평균 0·표준편차 1의 z-score로 변환

print(z1)  # 평균 0·표준편차 1로 표준화한 x1의 z점수 배열을 출력
print(z2)  # 평균 0·표준편차 1로 표준화한 x2의 z점수 배열을 출력

[-1.41421356 -0.70710678  0.          0.70710678  1.41421356]
[-1.41421356 -0.70710678  0.          0.70710678  1.41421356]


### 5-2. Min-Max Scaling

가장 작은 값을 0, 가장 큰 값을 1로 변환한다.

In [12]:
# 셀 목적: 각 값을 최솟값 0·최댓값 1 범위로 Min-Max 변환한다.
# 해석 포인트: 선형 배율만 다른 두 배열은 변환 후 같은 상대 위치를 갖는다.
# (값 - 최솟값) / (최댓값 - 최솟값)
minmax_x1 = (x1 - x1.min()) / (x1.max() - x1.min())  # x1을 최솟값 0·최댓값 1 범위로 변환
minmax_x2 = (x2 - x2.min()) / (x2.max() - x2.min())  # x2를 최솟값 0·최댓값 1 범위로 변환

print(minmax_x1)  # 최솟값 0·최댓값 1 범위로 변환한 x1의 최소-최대 정규화 결과를 출력
print(minmax_x2)  # 최솟값 0·최댓값 1 범위로 변환한 x2의 최소-최대 정규화 결과를 출력

[0.   0.25 0.5  0.75 1.  ]
[0.   0.25 0.5  0.75 1.  ]


### 5-3. scikit-learn을 이용한 스케일링

scikit-learn은 머신러닝과 데이터 전처리에 사용하는 라이브러리다.

In [13]:
# 셀 목적: 스케일이 서로 다른 두 변수를 열로 갖는 DataFrame을 만든다.
# 해석 포인트: 표 형태로 구성하면 scikit-learn 스케일러가 열별 최솟값·평균·표준편차를 학습한다.
# X1은 1~5, X2는 10~50인 표 형태 데이터
X = pd.DataFrame({  # 스케일이 다른 두 열을 가진 원본 DataFrame을 X에 저장
    "X1": [1, 2, 3, 4, 5],  # 첫 번째 열 X1에 1~5 범위 값을 지정
    "X2": [10, 20, 30, 40, 50]  # 두 번째 열 X2에 10~50 범위 값을 지정
})  # 딕셔너리 구성과 함수 호출을 함께 마침

X  # 생성한 원본 DataFrame을 노트북 출력으로 표시

,X1,X2
0,1,10
1,2,20
2,3,30
3,4,40
4,5,50


In [14]:
# 셀 목적: MinMaxScaler로 각 열을 0~1 범위로 변환한다.
# 해석 포인트: fit_transform()은 학습 데이터에서 최솟값·최댓값을 찾는 fit과 실제 변환 transform을 연속 수행한다.
from sklearn.preprocessing import MinMaxScaler  # 0~1 범위 변환 도구 MinMaxScaler를 불러옴

# 스케일러 객체 생성
scaler = MinMaxScaler()  # 0~1 변환을 수행할 MinMaxScaler 객체 생성

# 데이터의 최솟값과 최댓값을 확인하고 0~1 범위로 변환
Z = scaler.fit_transform(X)  # Min-Max 변환 결과를 Z에 저장

# 결과는 NumPy 배열이므로, 보기 좋게 DataFrame으로 변환
pd.DataFrame(Z, columns=["X1", "X2"])  # 배열 결과에 열 이름을 붙여 DataFrame 형태로 표시

,X1,X2
0,0.00,0.00
1,0.25,0.25
2,0.50,0.50
3,0.75,0.75
4,1.00,1.00


### 5-4. scikit-learn Standard Scaling

각 열의 평균을 0으로, 표준편차를 1로 맞추는 방식이다.

In [15]:
# 셀 목적: StandardScaler로 각 열을 평균 0·모표준편차 1에 가깝게 변환한다.
# 해석 포인트: 변환 결과는 NumPy 배열이므로 원래 열 이름을 붙여 DataFrame으로 확인한다.
from sklearn.preprocessing import StandardScaler  # 평균 0·표준편차 1 변환 도구 StandardScaler를 불러옴

# 표준 스케일링 도구 생성
standard_scaler = StandardScaler()  # 평균 0·표준편차 1 변환용 StandardScaler 객체 생성

# 각 열을 평균 0, 표준편차 1 기준으로 변환
S = standard_scaler.fit_transform(X)  # 표준화된 배열을 S에 저장

# 보기 좋게 DataFrame으로 변환
pd.DataFrame(S, columns=["X1", "X2"])  # 배열 결과에 열 이름을 붙여 DataFrame 형태로 표시

,X1,X2
0,-1.414214,-1.414214
1,-0.707107,-0.707107
2,0.000000,0.000000
3,0.707107,0.707107
4,1.414214,1.414214


## 6. 범위와 사분위 범위 계산

정규분포를 따라 무작위 데이터를 만든 뒤,
범위와 사분위 범위(IQR)를 계산한다.

In [16]:
# 셀 목적: 평균 100·표준편차 20인 정규분포에서 1,000개의 난수를 생성한다.
# 해석 포인트: 난수 시드를 고정하지 않았으므로 실행할 때마다 개별 값과 범위는 조금 달라질 수 있다.
# np.random.normal(평균, 표준편차, 데이터 개수)
x = np.random.normal(100, 20, size=1000)  # 현재 통계 개념을 확인할 실습 데이터를 x에 저장

# 데이터가 너무 많으므로 앞의 10개만 확인
print(x[:10])  # 범위와 IQR 계산에 사용할 난수 데이터의 앞 10개 값을 출력

[140.83475022  89.15883498  91.53207598 102.14010052  96.46652397
  67.65585693 137.56919424  92.97228693  99.50573138 115.18308432]


In [17]:
# 셀 목적: 데이터의 전체 범위를 np.ptp()와 최댓값-최솟값 방식으로 각각 계산한다.
# 해석 포인트: 범위는 양끝 두 값만 사용하므로 이상치와 표본크기에 민감하다.
# np.ptp는 최댓값 - 최솟값을 계산한다.
print(np.ptp(x))  # NumPy의 ptp()로 계산한 최댓값과 최솟값의 차이인 범위를 출력

# 범위를 직접 계산한 방식
print(np.max(x) - np.min(x))  # 최댓값에서 최솟값을 직접 빼 계산한 범위를 출력해 ptp() 결과와 비교

127.61957332965733
127.61957332965733


## 7. 사분위 범위(IQR) 계산

IQR은 3사분위수(Q3)에서 1사분위수(Q1)를 뺀 값이다.
이상치의 영향을 범위보다 덜 받는다.

In [18]:
# 셀 목적: NumPy 분위수와 SciPy 전용 함수로 IQR을 각각 계산한다.
# 해석 포인트: IQR은 중앙 50%의 폭인 Q3-Q1이므로 극단값의 영향을 범위보다 덜 받는다.
# Q3(75% 지점) - Q1(25% 지점)
iqr_by_numpy = np.quantile(x, 0.75) - np.quantile(x, 0.25)  # NumPy 분위수로 Q3-Q1을 계산

# SciPy 함수로 IQR 계산
iqr_by_scipy = stats.iqr(x)  # SciPy 전용 함수로 IQR을 계산

print(iqr_by_numpy)  # NumPy 백분위수로 계산한 사분위범위 Q3-Q1을 출력
print(iqr_by_scipy)  # SciPy의 iqr()로 계산한 사분위범위를 출력해 NumPy 결과와 비교

27.869278976792756
27.869278976792756
